In [7]:
import pandas as pd
import numpy as np
import json
import time
import os
import random
from sklearn.metrics import confusion_matrix
from xgboost import XGBClassifier

OUTPUT_DIR = "output"
OUTPUT_FILE = "xgboost.json"
MEMBER_NAME = "Robera Abajobir"
MODEL_NAME = "XGBoost"

np.random.seed(2)
random.seed(2)

def load_data():
    print("Loading uploaded data...")
    X_train = pd.read_csv('X_train.csv').values
    y_train = pd.read_csv('y_train.csv').values.flatten()
    X_test = pd.read_csv('X_test.csv').values
    y_test = pd.read_csv('y_test.csv').values.flatten()
    return X_train, y_train, X_test, y_test

def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    accuracy = (y_pred == y_test).sum() / len(y_test)
    return cm, accuracy

def main():
    X_train, y_train, X_test, y_test = load_data()
    num_classes = len(np.unique(y_train))

    # Balanced search space for CPU
    n_estimators_options = [100, 200, 300]
    max_depth_options = [3, 4, 5, 6]
    learning_rate_options = [0.02, 0.05, 0.1]
    subsample_options = [0.7, 0.85, 1.0]
    colsample_bytree_options = [0.7, 0.85, 1.0]
    reg_lambda_options = [0, 1, 5]
    reg_alpha_options = [0, 0.1, 0.5]

    NUM_TRIALS = 30

    trials_data = []
    best_accuracy_search = 0.0
    best_hyperparameters = {}

    print(f"Running {NUM_TRIALS} hyperparameter trials...")

    for i in range(NUM_TRIALS):
        params = {
            "n_estimators": random.choice(n_estimators_options),
            "max_depth": random.choice(max_depth_options),
            "learning_rate": random.choice(learning_rate_options),
            "subsample": random.choice(subsample_options),
            "colsample_bytree": random.choice(colsample_bytree_options),
            "reg_lambda": random.choice(reg_lambda_options),
            "reg_alpha": random.choice(reg_alpha_options),
            "n_jobs": -1
        }

        print(f"Trial {i+1}: {params}")

        model = XGBClassifier(
            objective="multi:softmax",
            num_class=num_classes,
            eval_metric="mlogloss",
            tree_method="hist",
            **params
        )

        model.fit(X_train, y_train)
        cm, accuracy = evaluate_model(model, X_test, y_test)

        trials_data.append({
            "hyperparameters": params,
            "confusion_matrix": cm.tolist(),
            "accuracy": float(accuracy)
        })

        if accuracy > best_accuracy_search:
            best_accuracy_search = accuracy
            best_hyperparameters = params

    print("Best accuracy from search:", best_accuracy_search)
    print("Best hyperparameters:", best_hyperparameters)

    best_hyperparameters["n_jobs"] = -1

    final_model = XGBClassifier(
        objective="multi:softmax",
        num_class=num_classes,
        eval_metric="mlogloss",
        tree_method="hist",
        **best_hyperparameters
    )

    start_train = time.time()
    final_model.fit(X_train, y_train)
    end_train = time.time()

    start_test = time.time()
    final_cm, final_accuracy = evaluate_model(final_model, X_test, y_test)
    end_test = time.time()

    output_data = {
        "model_name": MODEL_NAME,
        "person_name": MEMBER_NAME,
        "best_hyperparameters": best_hyperparameters,
        "best_confusion_matrix": final_cm.tolist(),
        "trials": trials_data,
        "total_train_time": round(end_train - start_train, 4),
        "total_test_time": round(end_test - start_test, 4)
    }

    if not os.path.exists(OUTPUT_DIR):
        os.makedirs(OUTPUT_DIR)

    output_path = os.path.join(OUTPUT_DIR, OUTPUT_FILE)
    with open(output_path, "w") as f:
        json.dump(output_data, f, indent=2)

    print("Saved JSON to:", output_path)

if __name__ == "__main__":
    main()


Loading uploaded data...
Running 30 hyperparameter trials...
Trial 1: {'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.02, 'subsample': 0.85, 'colsample_bytree': 0.7, 'reg_lambda': 5, 'reg_alpha': 0.5, 'n_jobs': -1}
Trial 2: {'n_estimators': 200, 'max_depth': 5, 'learning_rate': 0.1, 'subsample': 0.7, 'colsample_bytree': 1.0, 'reg_lambda': 0, 'reg_alpha': 0.5, 'n_jobs': -1}
Trial 3: {'n_estimators': 300, 'max_depth': 4, 'learning_rate': 0.05, 'subsample': 1.0, 'colsample_bytree': 0.85, 'reg_lambda': 5, 'reg_alpha': 0.5, 'n_jobs': -1}
Trial 4: {'n_estimators': 200, 'max_depth': 6, 'learning_rate': 0.1, 'subsample': 0.85, 'colsample_bytree': 0.7, 'reg_lambda': 0, 'reg_alpha': 0.1, 'n_jobs': -1}
Trial 5: {'n_estimators': 200, 'max_depth': 5, 'learning_rate': 0.05, 'subsample': 0.85, 'colsample_bytree': 1.0, 'reg_lambda': 0, 'reg_alpha': 0.5, 'n_jobs': -1}
Trial 6: {'n_estimators': 100, 'max_depth': 4, 'learning_rate': 0.02, 'subsample': 0.7, 'colsample_bytree': 0.7, 'reg_lambda': 